# 01 — Data QA

Run `python generate_data.py` from the repository root first. Generated event files are deterministic demo data; committed player, assignment and purchase files are retained as project inputs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
players = pd.read_csv(DATA / 'players.csv', parse_dates=['install_date'])
sessions = pd.read_csv(DATA / 'sessions.csv', parse_dates=['session_start'])
rounds = pd.read_csv(DATA / 'bingo_rounds.csv', parse_dates=['round_timestamp'])
assignments = pd.read_csv(DATA / 'ab_test_assignments.csv')
bonuses = pd.read_csv(DATA / 'bonus_events.csv', parse_dates=['bonus_timestamp'])
transactions = pd.read_csv(DATA / 'iap_transactions.csv', parse_dates=['transaction_timestamp'])
for name, frame in {'players':players,'sessions':sessions,'rounds':rounds,'assignments':assignments,'bonuses':bonuses,'transactions':transactions}.items():
    print(f'{name:14s} {frame.shape}')

## Integrity checks

QA/test accounts are excluded by the documented acquisition-channel flag, not by an outcome-dependent heuristic.

In [ ]:
assert players.player_id.is_unique
assert sessions.session_id.is_unique
assert rounds.round_id.is_unique
assert assignments.player_id.is_unique
assert sessions.session_length_sec.ge(0).all()
assert rounds[['coins_wagered','coins_won']].ge(0).all().all()
qa_ids = set(players.loc[players.acquisition_channel.eq('internal_qa'),'player_id'])
clean = players.loc[~players.acquisition_channel.eq('internal_qa')]
print(f'QA accounts: {len(qa_ids):,}; analysis players: {clean.player_id.nunique():,}')
assert not clean.player_id.isin(qa_ids).any()